# Assistente de Investimentos: RPA com Python e IA Generativa no N8N

Este notebook faz parte do Desafio DIO **Criando um Assistente de Investimentos com RPA e IA Generativa**.

### Pipeline da Automação:
1. **RPA (Robotic Process Automation):** Coleta dados de clientes em uma página web simulada usando Python e BeautifulSoup.
2. **Orquestração (N8N):** Envia os dados para um Webhook no N8N via requisição HTTP POST.
3. **Processamento Inteligente:** O N8N cruza o perfil e saldo dos clientes com o catálogo de investimentos (`data.csv`).
4. **Geração de Mensagens:** Gera recomendações personalizadas (templates no MVP e IA Generativa no desafio completo).

---

## 1. Instalação das Dependências
Precisamos do `requests` para requisições HTTP e do `beautifulsoup4` para o web scraping do HTML.

In [ ]:
!pip install requests beautifulsoup4

## 2. Extração dos Clientes via RPA (Web Scraping)
Acessamos a página hospedada no GitHub Pages e extraímos cada linha da tabela `#clientes`.

In [ ]:
import requests
from bs4 import BeautifulSoup

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"

print(f"Baixando página: {URL_PAGINA}...")
resposta = requests.get(URL_PAGINA, timeout=15)
resposta.raise_for_status()

soup = BeautifulSoup(resposta.text, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []
for linha in linhas:
    colunas = linha.find_all("td")
    if len(colunas) >= 4:
        cliente = {
            "nome": colunas[0].get_text(strip=True),
            "email": colunas[1].get_text(strip=True),
            "saldo": colunas[2].get_text(strip=True),
            "perfil": colunas[3].get_text(strip=True),
        }
        clientes.append(cliente)

print(f"Total de clientes extraídos: {len(clientes)}\n")
for c in clientes:
    print(f"- {c['nome']} ({c['perfil']}) | Saldo: {c['saldo']} | E-mail: {c['email']}")

## 3. Envio para o Webhook do N8N
Agora enviamos a lista de clientes para o workflow no N8N.

> 💡 **Como obter a URL do Webhook no N8N:**
> - No N8N, abra o nó **Webhook - Clientes**.
> - Para testar no editor: copie a **Test URL** (ex: `http://localhost:5678/webhook-test/assistente-investimentos`) e clique em *Listen for test event* no N8N antes de rodar a célula.
> - Para produção: ative o workflow (toggle superior direito) e copie a **Production URL** (ex: `http://localhost:5678/webhook/assistente-investimentos`).

In [ ]:
# Insira a URL do seu Webhook N8N abaixo:
N8N_WEBHOOK = "http://localhost:5678/webhook/assistente-investimentos"

payload = {"clientes": clientes}

print(f"Enviando payload com {len(clientes)} clientes para {N8N_WEBHOOK}...")
try:
    resp = requests.post(N8N_WEBHOOK, json=payload, headers={"Content-Type": "application/json"}, timeout=60)
    print(f"Status HTTP: {resp.status_code}")
    
    if resp.status_code == 200:
        resultado = resp.json()
        print("\n================ RETORNO DO N8N ================")
        total = resultado.get('total_processado') or len(resultado.get('resultados', []))
        print(f"Total de Recomendações Geradas: {total}\n")
        
        for i, item in enumerate(resultado.get('resultados', []), 1):
            print(f"[{i}] Cliente: {item.get('nome')} | Perfil: {item.get('perfil')}")
            print(f"Mensagem:")
            print(item.get('mensagem', ''))
            print("-" * 60)
    else:
        print(f"Erro retornado pelo N8N ({resp.status_code}):")
        print(resp.text)
except requests.exceptions.ConnectionError:
    print("[ERRO] Não foi possível conectar ao Webhook do N8N. Verifique se o n8n está rodando e a URL está correta.")
except Exception as e:
    print(f"[ERRO] Ocorreu uma exceção: {e}")